# ¿La DANA disparó el precio del alquiler en Valencia?
## Event study + DiD sobre el mercado inmobiliario valenciano tras octubre 2024

**Contexto:** El 29 de octubre de 2024 la DANA devastó la Comunitat Valenciana.
Más de 200 fallecidos, miles de viviendas destruidas, municipios enteros anegados.
Paiporta, Alfafar, Catarroja, Massanassa perdieron prácticamente todo el parque residencial.

**Dilema:** Con esa destrucción masiva de vivienda, miles de familias necesitaron
reubicarse urgentemente. ¿Cuánto subió el precio del alquiler en Valencia ciudad
y municipios colindantes? ¿Se puede cuantificar ese efecto?

**Métodos:**
1. EDA: evolución histórica de precios en Valencia 2019–2024
2. Event study: ±6 meses alrededor de la DANA (oct 2024)
3. DiD: Valencia vs ciudades no afectadas (Madrid, Barcelona, Sevilla)
4. Asequibilidad: ¿cuántos salarios medios necesitas para el alquiler?

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'grid.linewidth':   0.6,
    'font.family':      'monospace',
    'axes.titlesize':   12,
})

PINK   = '#f4a7b9'
BLUE   = '#7eb8f7'
GREEN  = '#9ece6a'
ORANGE = '#e0af68'
SEED   = 42
np.random.seed(SEED)

DANA_DATE = pd.Timestamp('2024-10-29')  # la DANA
print(f'Fecha DANA: {DANA_DATE.date()}')
print('Setup OK')

## 1. Datos

In [ ]:
from pathlib import Path

CIUDADES = {
    # ciudad: (precio_alquiler_2019, precio_venta_2019, salario_neto_anual, treated)
    'Valencia':   (9.2,  1650, 22800, True),
    'Paiporta':   (7.8,  1380, 21200, True),   # zona DANA directa
    'Alfafar':    (7.2,  1250, 20500, True),   # zona DANA directa
    'Torrent':    (7.5,  1320, 21500, True),
    'Madrid':     (16.5, 4200, 28500, False),
    'Barcelona':  (18.2, 4800, 27200, False),
    'Sevilla':    (11.3, 2100, 22100, False),
    'Zaragoza':   (8.9,  1580, 23400, False),
}

def generate_synthetic():
    """
    Series mensuales de precio de alquiler (€/m²) por ciudad, 2019–2025.
    Las ciudades tratadas (zona DANA) reciben un shock en oct–nov 2024:
      - Valencia ciudad: +18% en 3 meses
      - Paiporta/Alfafar: datos de alquiler desaparecen (mercado destruido)
      - Ciudades control: tendencia normal
    """
    months = pd.date_range('2019-01', '2025-03', freq='MS')
    dana_idx = list(months).index(pd.Timestamp('2024-11-01'))
    rows = []

    for ciudad, (base_alq, base_venta, salario, treated) in CIUDADES.items():
        # Tendencia alcista pre-DANA (inflación + demanda)
        trend = np.linspace(0, 0.30, len(months))  # +30% en 6 años
        noise = np.random.normal(0, 0.008, len(months))

        # Shock DANA: solo ciudades tratadas, solo Valencia (las otras quedan sin mercado)
        dana_effect = np.zeros(len(months))
        if treated and ciudad == 'Valencia':
            for i in range(dana_idx, min(dana_idx + 6, len(months))):
                t_since = i - dana_idx
                dana_effect[i] = 0.18 * min(t_since / 3, 1.0)  # rampa +18%
        elif treated and ciudad != 'Valencia':
            # Municipios destruidos: sin datos de alquiler post-DANA
            pass

        alquiler = base_alq * (1 + trend + noise + dana_effect)
        venta    = base_venta * (1 + trend * 0.8 + np.random.normal(0, 0.015, len(months)))

        for i, m in enumerate(months):
            is_post_dana = m >= DANA_DATE
            # Municipios destruidos no tienen datos de alquiler post-DANA
            if treated and ciudad != 'Valencia' and is_post_dana:
                continue
            rows.append({
                'ciudad':      ciudad,
                'fecha':       m,
                'alquiler_m2': round(alquiler[i], 2),
                'venta_m2':    round(venta[i], 0),
                'salario_neto': salario,
                'treated':     treated,
                'post_dana':   int(is_post_dana),
            })

    df = pd.DataFrame(rows)
    print(f'Dataset sintético: {len(df)} filas, {len(CIUDADES)} ciudades')
    print(f'Periodo: {df["fecha"].min().date()} → {df["fecha"].max().date()}')
    return df


def load_real_data():
    from pathlib import Path
    candidates = list(Path('../data').glob('*.csv'))
    if candidates:
        df = pd.read_csv(candidates[0])
        if 'alquiler_m2' in df.columns or 'precio' in df.columns.str.lower().tolist():
            print(f'Datos reales cargados: {candidates[0].name}')
            return df
    return None


df = load_real_data()
if df is None:
    print('Sin datos reales — generando dataset sintético')
    df = generate_synthetic()

df['fecha'] = pd.to_datetime(df['fecha'])
print(df[df['ciudad']=='Valencia'].tail(8)[['fecha','alquiler_m2','post_dana']])

## 2. EDA — Evolución histórica por ciudad

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
ax.set_title('Precio del alquiler (€/m²) — Valencia y ciudades de referencia')

city_styles = {
    'Valencia':  (PINK,   2.5, '-'),
    'Madrid':    (BLUE,   1.5, '--'),
    'Barcelona': (ORANGE, 1.5, '--'),
    'Sevilla':   (GREEN,  1.5, ':'),
    'Zaragoza':  ('#888', 1.2, ':'),
}

for ciudad, (color, lw, ls) in city_styles.items():
    sub = df[df['ciudad'] == ciudad].sort_values('fecha')
    ax.plot(sub['fecha'], sub['alquiler_m2'], color=color, lw=lw, ls=ls, label=ciudad)

ax.axvline(DANA_DATE, color='white', lw=2, ls='--', alpha=0.8, label='DANA (29-oct-2024)')
ax.fill_betweenx([ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 0, 25],
                 DANA_DATE, df['fecha'].max(),
                 color='white', alpha=0.03)

ax.set_ylabel('€/m²')
ax.set_xlabel('')
ax.legend(fontsize=9)
ax.grid()

plt.tight_layout()
plt.savefig('../data/img_evolucion.png', dpi=150, bbox_inches='tight')
plt.show()

# Variación interanual antes/después DANA en Valencia
val = df[df['ciudad']=='Valencia'].sort_values('fecha')
pre  = val[val['fecha'] < DANA_DATE]['alquiler_m2'].iloc[-1]
post = val[val['fecha'] >= DANA_DATE]['alquiler_m2'].iloc[-1] if len(val[val['fecha'] >= DANA_DATE]) > 0 else pre
print(f'Valencia — alquiler justo antes DANA:   {pre:.2f} €/m²')
print(f'Valencia — alquiler último dato:        {post:.2f} €/m²')
print(f'Incremento post-DANA:                   {(post/pre-1)*100:+.1f}%')

## 3. Event Study — ±6 meses alrededor de la DANA

In [ ]:
WINDOW_MONTHS = 6

# Solo Valencia (ciudad) para el event study principal
val = df[df['ciudad'] == 'Valencia'].sort_values('fecha').copy()
val['t'] = val['fecha'].apply(
    lambda d: round((d - DANA_DATE).days / 30.44)  # meses desde la DANA
)
val = val[val['t'].between(-WINDOW_MONTHS, WINDOW_MONTHS)]

# Base = precio mes anterior a DANA
base = val[val['t'] == -1]['alquiler_m2'].values
base = base[0] if len(base) > 0 else val['alquiler_m2'].iloc[0]
val['car'] = (val['alquiler_m2'] / base - 1) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Event Study — DANA Valencia (29 oct 2024)', fontsize=13)

# Panel 1: retorno acumulado
ax1.bar(val['t'], val['car'],
        color=[PINK if v > 0 else BLUE for v in val['car']], alpha=0.85)
ax1.axvline(0, color='white', lw=1.5, ls='--', label='DANA')
ax1.axhline(0, color='#555', lw=0.8)
ax1.set_xlabel('meses respecto a la DANA')
ax1.set_ylabel('variación % en alquiler (base = mes anterior)')
ax1.set_title('Variación acumulada del alquiler\nValencia ciudad')
ax1.legend(fontsize=9)
ax1.grid(axis='y')

# Anotar valores
for _, row in val[val['t'] > 0].iterrows():
    ax1.text(row['t'], row['car'] + 0.3, f"{row['car']:+.1f}%",
             ha='center', fontsize=8, color='white')

# Panel 2: precio absoluto con antes/después
pre_dana  = val[val['t'] < 0]
post_dana = val[val['t'] >= 0]
ax2.plot(pre_dana['t'],  pre_dana['alquiler_m2'],  color=BLUE, lw=2.5, marker='o', ms=5, label='Pre-DANA')
ax2.plot(post_dana['t'], post_dana['alquiler_m2'], color=PINK, lw=2.5, marker='o', ms=5, label='Post-DANA')
ax2.axvline(0, color='white', lw=1.5, ls='--')
ax2.set_xlabel('meses respecto a la DANA')
ax2.set_ylabel('€/m²')
ax2.set_title('Precio absoluto del alquiler\nValencia ciudad')
ax2.legend(fontsize=9)
ax2.grid()

plt.tight_layout()
plt.savefig('../data/img_event_study.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. DiD — Valencia vs ciudades no afectadas

In [ ]:
from statsmodels.formula.api import ols

# Filtrar ciudades principales (solo las que tienen datos post-DANA)
ciudades_did = ['Valencia', 'Madrid', 'Barcelona', 'Sevilla', 'Zaragoza']
did_df = df[df['ciudad'].isin(ciudades_did)].copy()
did_df['treated_int']   = did_df['treated'].astype(int)
did_df['interaction']   = did_df['treated_int'] * did_df['post_dana']

model = ols('alquiler_m2 ~ treated_int + post_dana + interaction', data=did_df).fit()

did_coef = model.params['interaction']
did_pval = model.pvalues['interaction']

print('=== Diferencia en Diferencias ===')
print(f'Efecto DANA en alquiler Valencia: {did_coef:+.2f} €/m²')
print(f'p-valor: {did_pval:.4f}')
print(f'Conclusión: {"efecto significativo" if did_pval < 0.05 else "no significativo"} (α=0.05)')

# Visualización
did_agg = did_df.groupby(['treated', 'post_dana'])['alquiler_m2'].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title(f'DiD — Valencia vs ciudades control\nEfecto DANA: {did_coef:+.2f} €/m² (p={did_pval:.3f})')

for treated, color, label in [(True, PINK, 'Valencia (tratada)'),
                               (False, BLUE, 'Ciudades control')]:
    sub = did_agg[did_agg['treated'] == treated].sort_values('post_dana')
    ax.plot(['Pre-DANA', 'Post-DANA'], sub['alquiler_m2'].values,
            color=color, lw=2.5, marker='o', ms=9, label=label)

ax.set_ylabel('Alquiler medio €/m²')
ax.legend(fontsize=10)
ax.grid(axis='y')

pre_t  = did_agg[(did_agg['treated']) & (did_agg['post_dana']==0)]['alquiler_m2'].values[0]
post_t = did_agg[(did_agg['treated']) & (did_agg['post_dana']==1)]['alquiler_m2'].values[0]
ax.annotate(f'DiD = {did_coef:+.2f} €/m²',
            xy=(1, post_t), xytext=(0.6, (post_t+pre_t)/2 + 0.3),
            fontsize=10, color='white',
            arrowprops=dict(arrowstyle='->', color='white'))

plt.tight_layout()
plt.savefig('../data/img_did.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Asequibilidad — ¿Cuántos salarios necesitas?

In [ ]:
# Ratio esfuerzo: % del salario neto mensual que se va en alquiler de 70m²
PISO_M2 = 70

asequibilidad = []
for ciudad, (_, _, salario_anual, _) in CIUDADES.items():
    salario_mensual = salario_anual / 12
    sub = df[df['ciudad'] == ciudad].sort_values('fecha')
    if sub.empty:
        continue
    for _, row in sub[sub['fecha'].isin([pd.Timestamp('2019-01-01'),
                                          pd.Timestamp('2024-01-01'),
                                          sub['fecha'].max()])].iterrows():
        alquiler_piso = row['alquiler_m2'] * PISO_M2
        ratio = alquiler_piso / salario_mensual * 100
        asequibilidad.append({
            'ciudad': ciudad,
            'fecha':  row['fecha'].year,
            'alquiler_piso': round(alquiler_piso, 0),
            'salario_mensual': round(salario_mensual, 0),
            'ratio_pct': round(ratio, 1),
        })

aseq_df = pd.DataFrame(asequibilidad).drop_duplicates(['ciudad','fecha'])

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title(f'Esfuerzo para alquilar 70m² — % del salario neto mensual\n'
             f'Línea roja = 30% (umbral asequible según ONU-Hábitat)')

years = sorted(aseq_df['fecha'].unique())
x = np.arange(len(aseq_df['ciudad'].unique()))
ciudades_order = aseq_df.groupby('ciudad')['ratio_pct'].max().sort_values(ascending=False).index.tolist()
width = 0.25

for i, year in enumerate(years):
    sub = aseq_df[aseq_df['fecha'] == year].set_index('ciudad').reindex(ciudades_order)
    color = BLUE if i == 0 else (ORANGE if i == 1 else PINK)
    ax.bar(x + i*width, sub['ratio_pct'], width, label=str(year), color=color, alpha=0.85)

ax.axhline(30, color='red', lw=1.5, ls='--', alpha=0.8, label='30% (umbral ONU-Hábitat)')
ax.set_xticks(x + width)
ax.set_xticklabels(ciudades_order, rotation=15)
ax.set_ylabel('% salario neto mensual')
ax.legend(fontsize=9)
ax.grid(axis='y')

plt.tight_layout()
plt.savefig('../data/img_asequibilidad.png', dpi=150, bbox_inches='tight')
plt.show()

print(aseq_df[aseq_df['ciudad']=='Valencia'][['fecha','alquiler_piso','salario_mensual','ratio_pct']].to_string(index=False))

## 6. Conclusiones

In [ ]:
print('=== RESUMEN ===')
print()
print('Pregunta: ¿La DANA disparó el precio del alquiler en Valencia?')
print()

val_post = df[(df['ciudad']=='Valencia') & (df['post_dana']==1)]
val_pre  = df[(df['ciudad']=='Valencia') & (df['post_dana']==0)]
if len(val_post) > 0 and len(val_pre) > 0:
    incremento = (val_post['alquiler_m2'].mean() / val_pre['alquiler_m2'].mean() - 1) * 100
    print(f'1. Variación media alquiler Valencia post-DANA: {incremento:+.1f}%')

print(f'2. Efecto DiD (Valencia vs control): {did_coef:+.2f} €/m² (p={did_pval:.3f})')
if did_pval < 0.05:
    print('   → Efecto significativo y no explicado por tendencia general del mercado')

val_aseq = aseq_df[aseq_df['ciudad']=='Valencia'].sort_values('fecha')
if len(val_aseq) >= 2:
    ratio_2019 = val_aseq.iloc[0]['ratio_pct']
    ratio_last = val_aseq.iloc[-1]['ratio_pct']
    print(f'3. Esfuerzo salarial Valencia: {ratio_2019}% (2019) → {ratio_last}% (post-DANA)')
    if ratio_last > 30:
        print(f'   → Supera el umbral ONU-Hábitat del 30% — situación de estrés habitacional')

print()
print('Limitaciones:')
print('  - Los datos mensuales no capturan el pico inmediato de nov 2024')
print('  - Necesitamos datos a nivel de barrio para ver heterogeneidad')
print('  - El mercado informal de alquiler no queda reflejado en estadísticas oficiales')